In [ ]:
import re
import subprocess
from   pathlib import Path

# Keep this block minimal and close to the original notebook style.
GAME                 = "antichess"
GENERATED_CODE_PATH  = f"outputs/{GAME}.py"

LLM_MODEL            = "openai-codex/gpt-5.5:xhigh"
TIMEOUT_SECONDS      = 600
OPEN_SPIEL_MAX_STEPS = 40
LEGAL_ACTION_LIMIT   = 8
COMPARE_SEED         = None
PREFERRED_MOVES      = []

CODE_PATH            = Path(GENERATED_CODE_PATH)
RESPONSE_PATH        = CODE_PATH.with_suffix(".md")


## LLM implementation

This optional cell generates code from the local prompt and rule files.


In [ ]:
try:
    import shutil

    if not LLM_MODEL:
        raise ValueError("Set LLM_MODEL")

    # Keep prompt/rules loading here, not in the settings block.
    prompt_text = Path("input/prompt.txt").read_text(encoding="utf-8")
    rules_text = Path("input/game_rules.txt").read_text(encoding="utf-8")
    full_prompt = prompt_text + "\n\nHier folgt die Spielanleitung:\n\n" + rules_text

    pi_path = shutil.which("pi") or shutil.which("pi.cmd")
    if pi_path is None:
        fallback = Path.home() / "AppData/Roaming/npm/pi.cmd"
        if not fallback.exists():
            raise FileNotFoundError("Could not find pi or pi.cmd")
        pi_path = str(fallback)

    result = subprocess.run(
        [pi_path, "-p", "--model", LLM_MODEL],
        input=full_prompt,
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or "pi call failed")

    CODE_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESPONSE_PATH.write_text(result.stdout, encoding="utf-8")

    match = re.search(r"```python\s*(.*?)```", result.stdout, re.IGNORECASE | re.DOTALL)
    if match is None:
        raise RuntimeError("No fenced python block found in the LLM response")

    CODE_PATH.write_text(match.group(1).strip() + "\n", encoding="utf-8")
except Exception as exc:
    print(f"LLM call failed: {exc}")


## OpenSpiel and generated game loading


In [ ]:
import importlib.util
import sys
import pyspiel

try:
    game = pyspiel.load_game(GAME)

    if not CODE_PATH.exists():
        raise FileNotFoundError(f"Generated code missing: {CODE_PATH}")

    spec = importlib.util.spec_from_file_location(GAME, CODE_PATH)
    module = importlib.util.module_from_spec(spec)
    if spec is None or spec.loader is None:
        raise RuntimeError("Could not load generated game module")
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    llm_game = module.Game()
except Exception as exc:
    print(f"Game loading failed: {exc}")


## Seeded random comparison


In [21]:
try:
    import random
    import time

    PIECE_MAP = {"B": "P", "S": "N", "L": "B", "T": "R", "D": "Q", "K": "K"}

    def side_by_side(left, right, width=56):
        left_lines = left.splitlines() or [""]
        right_lines = right.splitlines() or [""]
        height = max(len(left_lines), len(right_lines))
        left_lines += [""] * (height - len(left_lines))
        right_lines += [""] * (height - len(right_lines))
        return "\n".join(
            f"{left_line:<{width}} | {right_line}"
            for left_line, right_line in zip(left_lines, right_lines)
        )

    def open_visible_state(state):
        fen = state.to_string().split()
        board_part = fen[0]
        side = "white" if fen[1] == "w" else "black"

        board = [None] * 64
        for rank_i, row in enumerate(board_part.split("/")[::-1]):
            file_i = 0
            for char in row:
                if char.isdigit():
                    file_i += int(char)
                else:
                    board[rank_i * 8 + file_i] = ("w" if char.isupper() else "b") + char.upper()
                    file_i += 1

        return tuple(board), side

    def llm_visible_state(state):
        board = []
        for piece in state.board:
            if piece is None:
                board.append(None)
            else:
                board.append(piece[0] + PIECE_MAP[piece[1]])

        side = "white" if state.to_move == 0 else "black"
        return tuple(board), side

    def format_board(board):
        lines = ["   a  b  c  d  e  f  g  h"]
        for rank_i in range(7, -1, -1):
            row = []
            for file_i in range(8):
                piece = board[rank_i * 8 + file_i]
                row.append(piece if piece is not None else "..")
            lines.append(f"{rank_i + 1} " + " ".join(row))
        return "\n".join(lines)

    def panel(title, board, side):
        return f"{title}\nto move: {side}\n{format_board(board)}"

    def llm_apply(state, action):
        next_state = llm_game.apply_action(state, action)
        return state if next_state is None else next_state

    seed = COMPARE_SEED if COMPARE_SEED is not None else int(time.time() * 1000) % 1_000_000_000
    rng = random.Random(seed)

    print(f"compare seed: {seed}")
    print("reference: openspiel")
    print("state matching: visible board + side to move")

    open_state = game.new_initial_state()
    llm_state = llm_game.initial_state()
    history = []

    for step in range(OPEN_SPIEL_MAX_STEPS):
        open_board, open_side = open_visible_state(open_state)
        llm_board, llm_side = llm_visible_state(llm_state)
        open_terminal = open_state.is_terminal()
        llm_terminal = llm_game.is_terminal(llm_state)

        print("=" * 96)
        print(f"step {step}")
        if history:
            print("recent moves:", " ".join(history[-8:]))

        if (open_board, open_side) != (llm_board, llm_side):
            print("visible state mismatch")
            print(side_by_side(
                panel("reference", open_board, open_side),
                panel("generated", llm_board, llm_side),
            ))
            break

        if open_terminal != llm_terminal:
            print(f"terminal mismatch: reference={open_terminal} | generated={llm_terminal}")
            print(side_by_side(
                panel("reference", open_board, open_side),
                panel("generated", llm_board, llm_side),
            ))
            break

        print(side_by_side(
            panel("reference", open_board, open_side),
            panel("generated", llm_board, llm_side),
        ))

        if open_terminal:
            print(f"returns: reference={open_state.returns()} | generated={llm_game.returns(llm_state)}")
            break

        open_actions = list(open_state.legal_actions())
        llm_actions = list(llm_game.legal_actions(llm_state))
        matchable = []

        for open_action in open_actions:
            open_next = open_state.clone()
            open_next.apply_action(open_action)
            target = open_visible_state(open_next)

            llm_matches = []
            for llm_action in llm_actions:
                if llm_visible_state(llm_apply(llm_state, llm_action)) == target:
                    llm_matches.append(llm_action)

            if llm_matches:
                matchable.append((open_action, llm_matches))

        print(
            f"legal moves: reference={len(open_actions)} | generated={len(llm_actions)} | matching reference moves={len(matchable)}/{len(open_actions)}"
        )

        if len(matchable) != len(open_actions):
            matched_open = {
                open_state.action_to_string(open_state.current_player(), open_action)
                for open_action, _ in matchable
            }
            open_names = sorted(
                open_state.action_to_string(open_state.current_player(), action)
                for action in open_actions
            )
            print("unmatched reference sample:", [name for name in open_names if name not in matched_open][:LEGAL_ACTION_LIMIT])

        if not matchable:
            llm_names = sorted(llm_game.action_to_name(action) for action in llm_actions)
            print("generated sample:", llm_names[:LEGAL_ACTION_LIMIT])
            break

        if step < len(PREFERRED_MOVES):
            preferred = PREFERRED_MOVES[step]
            preferred_pairs = [
                (open_action, llm_matches)
                for open_action, llm_matches in matchable
                if open_state.action_to_string(open_state.current_player(), open_action) == preferred
            ]
            if not preferred_pairs:
                print(f"preferred move not matchable: {preferred}")
                break
            open_action, llm_matches = preferred_pairs[0]
        else:
            open_action, llm_matches = rng.choice(matchable)

        llm_action = rng.choice(llm_matches)
        open_name = open_state.action_to_string(open_state.current_player(), open_action)
        llm_name = llm_game.action_to_name(llm_action)
        history.append(open_name)

        print(f"chosen move: reference {open_name} | generated {llm_name}")

        open_state.apply_action(open_action)
        llm_state = llm_apply(llm_state, llm_action)
    else:
        print(f"stopped after {OPEN_SPIEL_MAX_STEPS} steps")
except NameError:
    print("Comparison failed: load the games first")
except Exception as exc:
    print(f"Comparison failed: {exc}")


compare seed: 850978861
reference: openspiel
state matching: visible board + side to move
step 0
reference                                                | generated
to move: white                                           | to move: white
   a  b  c  d  e  f  g  h                                |    a  b  c  d  e  f  g  h
8 bR bN bB bQ bK bB bN bR                                | 8 bR bN bB bQ bK bB bN bR
7 bP bP bP bP bP bP bP bP                                | 7 bP bP bP bP bP bP bP bP
6 .. .. .. .. .. .. .. ..                                | 6 .. .. .. .. .. .. .. ..
5 .. .. .. .. .. .. .. ..                                | 5 .. .. .. .. .. .. .. ..
4 .. .. .. .. .. .. .. ..                                | 4 .. .. .. .. .. .. .. ..
3 .. .. .. .. .. .. .. ..                                | 3 .. .. .. .. .. .. .. ..
2 wP wP wP wP wP wP wP wP                                | 2 wP wP wP wP wP wP wP wP
1 wR wN wB wQ wK wB wN wR                                | 1 wR wN wB wQ wK wB w

- im aktuellen stand gibt es am anfang volle deckung der referenzzüge
- die generierte implementierung läuft in den ersten schritten sichtbar synchron zur referenz
- ein sofortiger grundsätzlicher regelbruch ist damit aktuell nicht erkennbar
- der erste eindruck ist deshalb eher positiv als negativ
- trotzdem ist das nur ein früher plausibilitätscheck und noch kein belastbarer korrektheitsnachweis
- interessant werden vor allem spätere stellen, an denen die deckung kleiner wird oder die sichtbaren zustände auseinanderlaufen